# 🎯 TrackVision — Real-Time Webcam Tracking on Colab

This notebook captures **live webcam frames** from your browser, sends them to Colab's GPU,
runs the full **Faster R-CNN + Siamese ReID** tracking pipeline, and displays annotated frames
in real time.

## Prerequisites
1. **GPU Runtime**: `Runtime → Change runtime type → T4 / A100`
2. **Model weights** in Google Drive at `MyDrive/Deep_Learning/`:
   - `fasterrcnn_mot16_finetuned.pth`
   - `siamese_reid_mot16.pth`
   - `tracker_engine.py`
3. **Allow webcam** access when prompted by the browser

---
## Step 1 — GPU Check & Mount Drive

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU! Go to Runtime → Change runtime type → GPU")

from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 — Copy Files & Install Dependencies

In [ ]:
import shutil, os

DRIVE_DIR = "/content/drive/MyDrive/Deep_Learning"
WORK_DIR = "/content/trackvision"
os.makedirs(WORK_DIR, exist_ok=True)

for f in ["tracker_engine.py", "fasterrcnn_mot16_finetuned.pth", "siamese_reid_mot16.pth"]:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join(WORK_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  ✅ {f} ({os.path.getsize(dst)/1e6:.1f} MB)")
    else:
        print(f"  ❌ MISSING: {src}")

os.chdir(WORK_DIR)
print(f"\nWorking dir: {WORK_DIR}")

---
## Step 3 — Load Models on GPU

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)

from tracker_engine import load_models, Tracker, Siamese_Network
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import numpy as np
import cv2
from PIL import Image
from scipy.spatial.distance import cdist

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
detector, embed_model = load_models(DEVICE)

print(f"\n✅ Models loaded on {DEVICE}")
print(f"   Detector:  {sum(p.numel() for p in detector.parameters()):,} params")
print(f"   ReID:      {sum(p.numel() for p in embed_model.parameters()):,} params")

---
## Step 4 — Define Webcam Capture (JavaScript)

This cell defines a function that uses **browser JavaScript** to:
1. Request webcam permission
2. Capture a single frame
3. Encode it as base64 JPEG
4. Return it to Python

In [ ]:
from google.colab.output import eval_js
from IPython.display import display, HTML, Javascript, Image as IPImage
from base64 import b64decode, b64encode
import io

# ---- JavaScript that captures ONE webcam frame ----
WEBCAM_JS = """
async function captureFrame() {
    // Create video element if it doesn't exist
    if (!window._webcamVideo) {
        const video = document.createElement('video');
        video.style.display = 'none';
        document.body.appendChild(video);

        const stream = await navigator.mediaDevices.getUserMedia({
            video: { width: 640, height: 480 }
        });
        video.srcObject = stream;
        await video.play();

        // Wait for video to be ready
        await new Promise(resolve => {
            video.onloadedmetadata = resolve;
            if (video.readyState >= 2) resolve();
        });
        // Extra wait for first frame
        await new Promise(r => setTimeout(r, 500));

        window._webcamVideo = video;
        window._webcamCanvas = document.createElement('canvas');
    }

    const video = window._webcamVideo;
    const canvas = window._webcamCanvas;
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    canvas.getContext('2d').drawImage(video, 0, 0);

    return canvas.toDataURL('image/jpeg', 0.85);
}
"""

# Inject the JS
display(Javascript(WEBCAM_JS))


def capture_frame():
    """Capture one frame from webcam via browser JS → returns numpy RGB array."""
    data_url = eval_js('captureFrame()')
    # data_url format: 'data:image/jpeg;base64,/9j/4AAQ...'
    b64_str = data_url.split(',')[1]
    jpg_bytes = b64decode(b64_str)
    img = Image.open(io.BytesIO(jpg_bytes)).convert('RGB')
    return np.array(img)


# Quick test — capture one frame to verify webcam works
test_frame = capture_frame()
print(f"✅ Webcam working! Frame shape: {test_frame.shape}")

---
## Step 5 — Define Processing Helpers

In [ ]:
# ---- ReID transform (same as training) ----
_reid_tf = transforms.Compose([
    transforms.Resize((128, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ---- Config ----
DET_THRESH = 0.80
SIM_THRESH = 0.85
EMA_ALPHA = 0.90


def extract_embeddings(frame_rgb, boxes):
    """Crop detections and extract ReID embeddings."""
    H, W = frame_rgb.shape[:2]
    crops = []
    for x1, y1, x2, y2 in boxes:
        x1, y1 = max(0, int(x1)), max(0, int(y1))
        x2, y2 = min(W, int(x2)), min(H, int(y2))
        patch = frame_rgb[y1:y2, x1:x2]
        if patch.size == 0:
            patch = np.zeros((64, 32, 3), dtype=np.uint8)
        crops.append(_reid_tf(Image.fromarray(patch)))

    batch = torch.stack(crops).to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            embs = embed_model(batch)
    return torch.nn.functional.normalize(embs, dim=1).cpu().numpy()


def id_to_color(track_id):
    """Generate a distinct RGB colour for each track ID."""
    h = (track_id * 2654435761) & 0xFFFFFF
    return ((h >> 16) & 0xFF, (h >> 8) & 0xFF, h & 0xFF)


def process_frame(frame_rgb, tracker):
    """Run detection + tracking on a single frame. Returns annotated RGB frame."""
    # Detection
    img_t = TF.to_tensor(frame_rgb).to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            preds = detector([img_t])[0]

    scores = preds['scores'].cpu().numpy()
    boxes = preds['boxes'].cpu().numpy()
    boxes = boxes[scores >= DET_THRESH]

    # Tracking
    if len(boxes) > 0:
        embs = extract_embeddings(frame_rgb, boxes)
        tids = tracker.update(list(embs))
    else:
        tids = []

    # Annotate (draw on a copy)
    annotated = frame_rgb.copy()
    for box, tid in zip(boxes, tids):
        x1, y1, x2, y2 = map(int, box)
        color = id_to_color(tid)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

        label = f'ID:{tid}'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(annotated, (x1, y1 - th - 8), (x1 + tw + 6, y1), color, cv2.FILLED)
        cv2.putText(annotated, label, (x1 + 3, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

    return annotated, len(boxes), len(tracker.gallery)


print("✅ Processing helpers ready")

---
## Step 6 — 🔴 LIVE Webcam Tracking

Run this cell to start the live tracking loop.

- A live video feed will appear below with bounding boxes and track IDs
- Stats (FPS, detections, unique IDs) are shown above each frame
- **To stop**: click the `Stop` button (⬛) in the toolbar, or press `Ctrl+M I` to interrupt

> **Tip:** The display updates in-place so it looks like a video stream!

In [ ]:
import time
from IPython.display import display, clear_output

# ---- Initialise tracker ----
tracker = Tracker(sim_threshold=SIM_THRESH, ema_alpha=EMA_ALPHA)
frame_count = 0
start_time = time.time()

# ---- Display area ----
print("🔴 LIVE TRACKING — press Stop (⬛) to end\n")

try:
    while True:
        # 1. Capture frame from browser webcam
        frame_rgb = capture_frame()

        # 2. Process: detect → embed → track → annotate
        annotated, num_dets, num_ids = process_frame(frame_rgb, tracker)
        frame_count += 1

        # 3. Calculate FPS
        elapsed = time.time() - start_time
        fps = frame_count / elapsed if elapsed > 0 else 0

        # 4. Add stats overlay to frame
        stats_text = f"FPS: {fps:.1f} | Detections: {num_dets} | Tracked IDs: {num_ids} | Frame: {frame_count}"
        cv2.rectangle(annotated, (0, 0), (len(stats_text) * 11, 32), (0, 0, 0), cv2.FILLED)
        cv2.putText(annotated, stats_text, (8, 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 200), 2, cv2.LINE_AA)

        # 5. Display — encode to JPEG and show inline
        _, jpg = cv2.imencode('.jpg', cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR),
                             [cv2.IMWRITE_JPEG_QUALITY, 80])

        clear_output(wait=True)
        print(f"🔴 LIVE TRACKING — press Stop (⬛) to end")
        print(f"   {stats_text}\n")
        display(IPImage(data=jpg.tobytes(), format='jpeg'))

except KeyboardInterrupt:
    pass

# ---- Summary ----
total_time = time.time() - start_time
print(f"\n\n{'='*50}")
print(f"🏁 Tracking stopped")
print(f"   Frames processed: {frame_count}")
print(f"   Total time:       {total_time:.1f}s")
print(f"   Average FPS:      {frame_count/total_time:.2f}")
print(f"   Unique IDs seen:  {len(tracker.gallery)}")
print(f"{'='*50}")

---
## 🧹 Cleanup (Optional)

Run this to release the webcam and free GPU memory.

In [ ]:
# Release webcam
eval_js("""
if (window._webcamVideo) {
    window._webcamVideo.srcObject.getTracks().forEach(t => t.stop());
    window._webcamVideo.remove();
    delete window._webcamVideo;
    delete window._webcamCanvas;
}
""")

# Free GPU memory
del detector, embed_model
torch.cuda.empty_cache()

print("✅ Webcam released, GPU memory freed.")